**Problem Statement:**

The goal of this project is to build a recommendation system that suggests what a user should read next based on their reading behavior. Given a dataset containing information about book chapters and user interactions, the system should recommend either the next chapter of the current book or suggest new books when a user finishes reading a book.●	Handle users or books with little to no history (cold start)

**Approach:**

*   The system first looks the user current reading state. It checks whether the user finshed book or not
*   If the user is in the middle of a book, the system simply recommends the next chapter.
*   This is because reading a book is sequential, so the next chapter is the most relevant suggestion.
*   If the user has finished a book, the system shifts to recommending new books.
*   It uses user behavior by finding what other users read after finishing the same book
*   It also uses genre similarity to recommend books with similar themes or categories.
*   This ensures recommendations are relevant even when user behavior data is limited.
*   Both behavior-based and genre-based recommendations are combined to get better results
*   For new users with no history, the system recommends popular books from different genres

**Evaluating Approach:**

To evaluate the recommendation system, we focus on situations where a user has finished reading a book, since this is when the system suggests new books. For each user, we identify chapters that are the last chapter of a book and use them as test points. At these points, we check what books the user actually read before and after that book, which serve as the ground truth

Tradeoffs considerd: **bold text**

1. Book Transition vs Genre Similarity :
Book transitions capture real user behavior but fail when there is no historical data.
Genre similarity works for new or less-read books but may be less personalized.

2. Cold Start Using Popularity vs Personalization :
For new users, we recommend popular books across genres.
This ensures good initial recommendations but lacks user-specific personalization.

**Future Improvement:**
Combining Results Without Ranking :
 Merged results from different methods by intersection and we can modify by ranking to results and take top 5


















steps to Run:
1. upload the two datasets and paste path in below code snippet
2. install dependencies in below cell

In [12]:


import pandas as pd
from collections import defaultdict
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity
import random

chapters = pd.read_csv("/content/chapters.csv")
interactions = pd.read_csv("/content/interactions.csv")

print("Chapters:", chapters.shape)
print("Interactions:", interactions.shape)

print(chapters.head())
print(interactions.head())

Chapters: (50000, 6)
Interactions: (1000000, 3)
   chapter_id  chapter_sequence_no  book_id  author_id published_date  \
0     2812946                    1   139726      66847     1990-03-22   
1     4330764                    2   139726      66847     1990-04-09   
2     2664499                    3   139726      66847     1990-04-07   
3     2260666                    4   139726      66847     1990-05-18   
4     6069976                    1   191772      62262     2008-07-30   

                                       tags  
0                            Fantasy|Horror  
1      Fantasy|Young Adult|Literary Fiction  
2                                   Fantasy  
3                  Literary Fiction|Fantasy  
4  Horror|Young Adult|Romance|Graphic Novel  
        user_id  chapter_id  book_id
0  user_2378720     5894067   444295
1  user_2321122     2532511   785684
2  user_2335775     6777764   999595
3  user_7906001     7366896   748410
4  user_9981689     7853186   418083


Since there is no timestamp or reading order of user, we assume the data order is the reading order. We also create shortcuts to quickly find which book a chapter belongs to.

In [13]:

interactions = interactions.reset_index()
interactions = interactions.sort_values(["user_id", "index"])

chapter_to_book = dict(zip(chapters["chapter_id"], chapters["book_id"]))
chapter_to_seq = dict(zip(chapters["chapter_id"], chapters["chapter_sequence_no"]))

book_max_seq = chapters.groupby("book_id")["chapter_sequence_no"].max().to_dict()



user_sequences = interactions.groupby("user_id")["chapter_id"].apply(list)
user_books = interactions.groupby("user_id")["book_id"].apply(list)

def remove_duplicates(seq):
    return [x for i, x in enumerate(seq) if i == 0 or x != seq[i-1]]

user_books = user_books.apply(remove_duplicates)

If a user is reading a book, we simply suggest the next chapter in that book if it is not the last chapter. This helps users continue reading smoothly.

In [14]:
chapters_sorted = chapters.sort_values(["book_id", "chapter_sequence_no"])

next_chapter_map = {}
for book_id, group in chapters_sorted.groupby("book_id"):
    ch_list = group["chapter_id"].tolist()
    for i in range(len(ch_list)-1):
        next_chapter_map[ch_list[i]] = ch_list[i+1]


 Learning how users move from one book to another based on their reading history. For each user, we observe the sequence of books they read and count how often one book is followed by another. This helps us suggest what to read next.

In [15]:
book_transitions = defaultdict(lambda: defaultdict(int))

for seq in user_books:
    for a, b in zip(seq, seq[1:]):
        book_transitions[a][b] += 1

Comparing books based on their genres . Books with similar genres are recommended together using cosine similarity

In [16]:
chapters["tags_list"] = chapters["tags"].str.split("|")
book_genres = chapters.groupby("book_id")["tags_list"].sum()

mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(book_genres)

similarity_matrix = cosine_similarity(genre_matrix)
book_index = {book_id: idx for idx, book_id in enumerate(book_genres.index)}

In [17]:

def is_last_chapter(chapter_id):
    book = chapter_to_book.get(chapter_id)
    return chapter_to_seq.get(chapter_id) == book_max_seq.get(book)

def recommend_next_chapter(chapter_id):
    return next_chapter_map.get(chapter_id)

def recommend_next_book(book_id, k=5):
    next_books = book_transitions.get(book_id, {})
    sorted_books = sorted(next_books.items(), key=lambda x: -x[1])
    return [int(b) for b, _ in sorted_books[:k]]

def recommend_by_genre_cosine(book_id, k=5):
    idx = book_index.get(book_id)
    if idx is None:
        return []

    scores = similarity_matrix[idx]
    similar_books = sorted(enumerate(scores), key=lambda x: -x[1])

    result = []
    for i, _ in similar_books[1:]:
        result.append(int(book_genres.index[i]))
        if len(result) == k:
            break
    return result


If a new user has no history, we recommend popular books from different genres based on given datset so it can diverse.

In [18]:

def recommend_cold_start(k_per_genre=1, max_genres=5):

    book_popularity = interactions["book_id"].value_counts()

    temp = chapters.copy()
    temp["tags_list"] = temp["tags"].str.split("|")
    temp = temp.explode("tags_list")

    temp["popularity"] = temp["book_id"].map(book_popularity)
    temp = temp.dropna(subset=["popularity"])

    temp = temp.sort_values(["tags_list", "popularity"], ascending=[True, False])

    top_per_genre = temp.groupby("tags_list").head(k_per_genre)

    result = top_per_genre["book_id"].drop_duplicates().head(max_genres)

    return result.astype(int).tolist()

Combination of all above methods to give recommendations. Users get both next chapters and new books. we are recommending chapters if its not last and new books based on last chapter books

In [19]:

def recommend_for_user(user_id):

    seq = user_sequences.get(user_id, [])


    if not seq:
        return {
            "continue_reading": [],
            "new_books": recommend_cold_start()
        }

    # Track latest chapter per book
    book_latest = {chapter_to_book.get(ch): ch for ch in seq}

    continue_recs = []
    completed_books = []

    for book, ch in book_latest.items():
        if not is_last_chapter(ch):
            next_ch = recommend_next_chapter(ch)
            if next_ch:
                continue_recs.append(int(next_ch))
        else:
            completed_books.append(book)

    new_book_recs = []

    for book in completed_books:
        seq_books = recommend_next_book(book)
        genre_books = recommend_by_genre_cosine(book)
        new_book_recs.extend(seq_books + genre_books)

    new_book_recs = list({int(b) for b in new_book_recs})

    return {
        "continue_reading": continue_recs[:5],
        "new_books": new_book_recs[:5]
    }


def main():

    test_user = random.choice(list(user_sequences.keys()))

    print("User:", test_user)

    result = recommend_for_user(test_user)

    print("\n--- Recommendations ---")
    print("Continue Reading:", result.get("continue_reading", []))
    print("New Books:", result.get("new_books", []))


if __name__ == "__main__":
    main()

User: user_0844673

--- Recommendations ---
Continue Reading: [2630862, 9741548, 6166429, 7391108]
New Books: [719489, 129411, 203782, 643719, 453907]


**EVALUATING:**

Creating test case of users reasing last chapter of book and ground truth as user other books and we are not condering users cse with middle of book because it is rulr based logic give next chapter with same book

In [32]:
#CREATE TEST CASES

def create_last_chapter_test_cases(user_sequences):

    test_cases = []

    for user_id, seq in user_sequences.items():

        for ch in seq:

            if is_last_chapter(ch):

                book = chapter_to_book.get(ch)

                test_cases.append((user_id, ch, book))

    return test_cases



def get_user_books(user_id):

    seq = user_sequences.get(user_id, [])
    books = []

    for ch in seq:
        b = chapter_to_book.get(ch)
        if not books or books[-1] != b:
            books.append(b)

    return books


def predict_next_books(book_id, k=5):

    seq_books = recommend_next_book(book_id, k)
    genre_books = recommend_by_genre_cosine(book_id, k)

    return list({int(b) for b in (seq_books + genre_books)})[:k]



import random

all_test_cases = create_last_chapter_test_cases(user_sequences)

test_20 = random.sample(all_test_cases, 20)

print("\n--- TEST RESULTS ---\n")

correct = 0

for i, (user_id, chapter_id, book_id) in enumerate(test_20, 1):

    books = get_user_books(user_id)

    if book_id in books:
        idx = books.index(book_id)
    else:
        continue

    actual_books = []

    if idx > 0:
        actual_books.append(books[idx - 1])  # previous

    if idx < len(books) - 1:
        actual_books.append(books[idx + 1])

    predicted = predict_next_books(book_id, k=5)

    is_correct = any(b in predicted for b in actual_books)

    if is_correct:
        correct += 1


accuracy = correct / len(test_20)
print("Correct Predictions:", correct)
print("Total Test Cases:", len(test_20))
print("Accuracy:", round(accuracy, 4))


--- TEST RESULTS ---


--- SUMMARY ---
Correct Predictions: 3
Total Test Cases: 20
Accuracy: 0.6
